In [ ]:
# Check if the links are URL parameters (different from the home page)
if documents:
    first_doc = documents[0]
    first_url = first_doc.url

    print(f"Home page URL: {first_url}\n")

    # Count unique base URLs (without parameters)
    unique_base_urls = set()
    for link in first_doc.internal_links:
        # Extract base URL before '?'
        base_url = link.url.split("?")[0]
        unique_base_urls.add(base_url)

    print(f"Unique base URLs found: {len(unique_base_urls)}")
    for base_url in sorted(unique_base_urls)[:5]:
        print(f"  - {base_url}")

    if len(unique_base_urls) > 5:
        print(f"  ... and {len(unique_base_urls) - 5} more")

    # Check if links are mostly variations of the home page
    home_base = first_url.split("?")[0]
    links_to_home_page = sum(
        1 for link in first_doc.internal_links if link.url.split("?")[0] == home_base
    )

    print(f"\nLinks to same base page: {links_to_home_page}")
    print(
        f"Links to different pages: {len(first_doc.internal_links) - links_to_home_page}"
    )

In [ ]:
# Show all the internal links found but not explored
if documents:
    first_doc = documents[0]
    print("Internal links found on the first page:")
    print(f"Total internal links: {len(first_doc.internal_links)}\n")

    for i, link in enumerate(first_doc.internal_links[:10], 1):
        print(f"{i}. {link.url}")

    if len(first_doc.internal_links) > 10:
        print(f"\n... and {len(first_doc.internal_links) - 10} more links")

## Diagnostic: Debug Link Discovery

Let's investigate why the 55 links found on the first page weren't crawled.

# Municipal Tax Assessor Web Crawling
## Woonsocket, RI Property Data Analysis

This notebook demonstrates crawling a municipal tax assessor GIS website to extract property data and analyze property values by street and neighborhood.

In [4]:
import logging
from collections import Counter

from WebCrawler.Serializers import Serializers
from WebCrawler.Spider import Spider

# Configure logging to track crawler activity
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
)

In [10]:
# Initialize and run the spider with progress tracking
# This crawler fetches street pages and their linked property pages
spider = Spider(
    start_url="https://gis.vgsi.com/WoonsocketRI/Streets.aspx",
    max_depth=3,  # Level 1: street pages, Level 2: individual property pages
    debug=False,
    show_progress=True,  # Enable real-time progress bar with visited/pending counts
    cache_dir=".assessor_cache",  # Cache responses to speed up repeat runs
)

# Run the crawl asynchronously
documents = await spider.run_async()
print(f"\n✓ Crawl complete: {len(documents)} pages fetched")

2026-06-08 08:51:14,697 WebCrawler.Spider DEBUG    Spider initialized: strategy=BFS, max_depth=3
2026-06-08 08:51:14,697 WebCrawler.Spider DEBUG    Spider initialized: strategy=BFS, max_depth=3
2026-06-08 08:51:14,697 WebCrawler.Spider DEBUG Spider initialized: strategy=BFS, max_depth=3
Crawling: 0 URLs [00:00, ? URLs/s]2026-06-08 08:51:14,704 WebCrawler.Spider DEBUG    Cache hit for https://gis.vgsi.com/WoonsocketRI/Streets.aspx
2026-06-08 08:51:14,704 WebCrawler.Spider DEBUG    Cache hit for https://gis.vgsi.com/WoonsocketRI/Streets.aspx
2026-06-08 08:51:14,704 WebCrawler.Spider DEBUG    Cache hit for https://gis.vgsi.com/WoonsocketRI/Streets.aspx
2026-06-08 08:51:14,704 WebCrawler.Spider DEBUG Cache hit for https://gis.vgsi.com/WoonsocketRI/Streets.aspx
2026-06-08 08:51:14,707 WebCrawler.Spider INFO     Visited: https://gis.vgsi.com/WoonsocketRI/Streets.aspx (Total Visited: 1)
2026-06-08 08:51:14,707 WebCrawler.Spider INFO     Visited: https://gis.vgsi.com/WoonsocketRI/Streets.aspx 


✓ Crawl complete: 1 pages fetched


In [11]:
# Display basic statistics about the crawl
print("=" * 60)
print("CRAWL STATISTICS")
print("=" * 60)
print(f"Total pages crawled: {len(documents)}")
print(f"Total links found: {sum(len(doc.links) for doc in documents)}")
print(f"  - Internal links: {sum(len(doc.internal_links) for doc in documents)}")
print(f"  - External links: {sum(len(doc.external_links) for doc in documents)}")
print()

# Show pages by HTTP status
status_counts = Counter(doc.status_code for doc in documents)
for status in sorted(status_counts.keys()):
    print(f"HTTP {status}: {status_counts[status]} pages")

CRAWL STATISTICS
Total pages crawled: 1
Total links found: 55
  - Internal links: 55
  - External links: 0

HTTP 200: 1 pages


In [12]:
# Display page details with formatted output
print("\n" + "=" * 60)
print("PAGES CRAWLED")
print("=" * 60)
for i, doc in enumerate(documents, 1):
    title = doc.title if doc.title else "(no title)"
    internal_count = len(doc.internal_links)
    external_count = len(doc.external_links)
    print(f"\n[{i}] {title}")
    print(f"    URL: {doc.url}")
    print(f"    Status: HTTP {doc.status_code}")
    print(f"    Links: {internal_count} internal, {external_count} external")


PAGES CRAWLED

[1] Vision Government Solutions
    URL: https://gis.vgsi.com/WoonsocketRI/Streets.aspx
    Status: HTTP 200
    Links: 55 internal, 0 external


In [13]:
# Export to Pandas DataFrame for analysis
serializer = Serializers(documents)
df = serializer.to_pandas(include_html=False)

print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nFirst 10 rows:")
display(df.head(10))


DataFrame shape: (55, 7)
Columns: ['url', 'title', 'status_code', 'domain', 'link_url', 'link_text', 'link_type']

First 10 rows:


,url,title,status_code,domain,link_url,link_text,link_type
0,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Search.aspx,Search,internal
1,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Sales.aspx,Sales Search,internal
2,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Feedback.aspx,Feedback,internal
3,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Default.aspx...,Home,internal
4,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Streets.aspx...,A,internal
5,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Streets.aspx...,B,internal
6,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Streets.aspx...,C,internal
7,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Streets.aspx...,D,internal
8,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Streets.aspx...,E,internal
9,https://gis.vgsi.com/WoonsocketRI/Streets.aspx,Vision Government Solutions,200,vgsi,https://gis.vgsi.com/WoonsocketRI/Streets.aspx...,F,internal


In [9]:
# Analyze property data: link distribution
print("\n" + "=" * 60)
print("LINK ANALYSIS")
print("=" * 60)

# Link type distribution
link_types = df["link_type"].value_counts()
print("\nLink type distribution:")
for link_type, count in link_types.items():
    print(f"  {link_type}: {count}")

# Pages by domain (if multiple domains crawled)
if len(df) > 0:
    unique_domains = df["domain"].nunique()
    print(f"\nUnique domains: {unique_domains}")
    if unique_domains > 1:
        print("\nPages by domain:")
        domain_counts = df["domain"].value_counts()
        for domain, count in domain_counts.items():
            print(f"  {domain}: {count} pages")


LINK ANALYSIS

Link type distribution:
  internal: 55

Unique domains: 1


In [ ]:
# Export results to files for further analysis or archival
import os
from datetime import datetime

os.makedirs("data", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Export to JSON (preserves nested link structure)
json_file = f"data/woonsocket_tax_assessor_{timestamp}.json"
serializer.to_json(json_file, include_html=False)
print(f"✓ Exported to JSON: {json_file}")

# Export to CSV (flattened format, one row per link)
csv_file = f"data/woonsocket_tax_assessor_{timestamp}.csv"
df.to_csv(csv_file, index=False)
print(f"✓ Exported to CSV: {csv_file}")

# Summary of exported data
print(f"\nExported {len(documents)} pages with {len(df)} link entries")